# Deep Agents 1주차 Walkthrough — 코드 4 데모

> Quickstart · 모델 swap (문자열) · 모델 swap (객체) · 커스텀 system_prompt
>
> 교안: [`archives/source/01_textbook.md`](../archives/source/01_textbook.md)

이 노트북은 **코드 실행이 핵심** 이다. 이론·그림·표는 교안에서 다루고, 여기서는 셀별로 무엇을 호출하는지 + 인라인 주석으로 핵심만 짚는다.

**섹션 지도**

| § | 데모 | 원본 스크립트 |
|---|---|---|
| §1 | 환경 점검 — `.env` 로드, 버전 확인, imports | (셋업) |
| §2 | **Demo 1** — Quickstart 리서치 에이전트 (5단계) | `01_quickstart_research_agent.py` |
| §3 | **Demo 2** — 모델 swap, 길 1 (`provider:model` 문자열) | `02_model_string_swap.py` |
| §4 | **Demo 3** — 모델 swap, 길 2 (LangChain 객체 + Ollama) | `03_model_object_ollama.py` |
| §5 | **Demo 4** — 커스텀 `system_prompt` + BASE_AGENT_PROMPT 인스펙션 | `04_custom_system_prompt.py` |
| §6 | (옵션) LangSmith 트레이싱 환경변수 | — |


## §1. 환경 점검 — Setup


In [1]:
# 1) 핵심 라이브러리 버전을 한 번 확인한다.
#    deepagents / langchain / langgraph / tavily / dotenv / ollama 모두 설치돼 있어야 한다.
#    (직전 세션에서 .venv/bin/pip install -r requirements.txt 로 설치 완료)

import importlib.metadata as meta

for pkg in [
    "deepagents",
    "langchain",
    "langchain-core",
    "langgraph",
    "langchain-openai",
    "langchain-ollama",
    "tavily-python",
    "python-dotenv",
]:
    try:
        version = meta.version(pkg)
    except meta.PackageNotFoundError:
        version = "MISSING"
    print(f"{pkg:24s}  {version}")


deepagents                0.5.6
langchain                 1.2.17
langchain-core            1.3.2
langgraph                 1.1.10
langchain-openai          1.2.1
langchain-ollama          1.1.0
tavily-python             0.7.24
python-dotenv             1.2.2


In [2]:
# 2) .env 파일 위치를 확인하고 환경변수를 메모리에 적재한다.
#    find_dotenv() 가 현재 작업 디렉토리에서 위로 올라가며 .env 를 찾는다.
#    노트북이 scripts/ 에 있으니 한 단계 위(week1-overview-taeyoung/)의 .env 가 잡힌다.

import os
from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv()
print(f".env 위치: {dotenv_path}")

loaded = load_dotenv(dotenv_path)
print(f"load_dotenv 결과: {loaded}\n")

# 어떤 키들이 잡혔는지 (값은 마스킹) 확인.
for key in [
    "OPENAI_API_KEY",
    "OPENAI_BASE_URL",
    "DEEPAGENT_MODEL",
    "TAVILY_API_KEY",
    "OLLAMA_MODEL",
]:
    val = os.environ.get(key, "")
    if not val:
        masked = "(empty)"
    elif "URL" in key or "MODEL" in key:
        masked = val  # URL/모델명은 그대로 보여도 안전
    else:
        masked = val[:6] + "..." + val[-3:] if len(val) > 12 else "***"
    print(f"  {key:18s} = {masked}")


.env 위치: /media/restful3/data/workspace/langchain-docs/deep-agents/week1-overview-taeyoung/.env
load_dotenv 결과: True

  OPENAI_API_KEY     = your-a...y-1
  OPENAI_BASE_URL    = http://localhost:8317/v1
  DEEPAGENT_MODEL    = gpt-5.4-mini
  TAVILY_API_KEY     = tvly-d...Kn7
  OLLAMA_MODEL       = PetrosStav/gemma3-tools:12b


In [3]:
# 3) 핵심 import.
#    이 4개가 깨끗이 import 되면 본 노트북의 모든 데모가 실행 가능한 상태다.

from langchain.chat_models import init_chat_model  # 모델 인스턴스화 진입점
from deepagents import create_deep_agent           # 라이브러리 메인 팩토리
from tavily import TavilyClient                    # Demo 1 의 검색 클라이언트

print("import 완료")
print("create_deep_agent =", create_deep_agent)


import 완료
create_deep_agent = <function create_deep_agent at 0x78c428357380>


## §2. Demo 1 — Quickstart 리서치 에이전트

원본: [`scripts/01_quickstart_research_agent.py`](../archives/scripts_py/01_quickstart_research_agent.py) · 교안: §3.2\~§3.4

다섯 단계 — ① 모델 인스턴스화 → ② 검색 도구 정의 → ③ `create_deep_agent` 호출 → ④ `agent.invoke()` → ⑤ 결과에서 4대 능력 흔적 찾기.


In [4]:
# §2.1 — 모델 인스턴스화.
#
# init_chat_model 의 두 형식:
#   1) 문자열  : init_chat_model("openai:gpt-5.4-mini")            ← 이 셀
#   2) 객체    : init_chat_model(model="...", model_provider=...)  ← Demo 3

import os
from langchain.chat_models import init_chat_model

MODEL_NAME = os.environ.get("DEEPAGENT_MODEL", "gpt-4o-mini")
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL")  # None 이면 OpenAI 공식 엔드포인트

# OPENAI_BASE_URL 이 채워져 있으면 OpenAI 호환 프록시(clipproxyapi 등)로 라우팅된다.
# 비어 있으면 base_url 인자를 아예 안 넘겨 OpenAI 공식으로 간다.
extra = {"base_url": OPENAI_BASE_URL} if OPENAI_BASE_URL else {}

# "openai:gpt-..." — 콜론 앞이 provider, 뒤가 model 식별자.
# init_chat_model 이 이 문자열을 파싱해 적절한 ChatModel 클래스(여기선 ChatOpenAI)를 고른다.
model = init_chat_model(f"openai:{MODEL_NAME}", **extra)

print(f"모델: openai:{MODEL_NAME}")
print(f"base_url: {OPENAI_BASE_URL or '(OpenAI 공식)'}")
print(f"인스턴스: {type(model).__name__}")


모델: openai:gpt-5.4-mini
base_url: http://localhost:8317/v1
인스턴스: ChatOpenAI


In [5]:
# §2.2 — Tavily 검색 도구 정의.
#
# Tavily 클라이언트를 함수로 감싸기만 하면 deepagents 가 바로 도구로 쓸 수 있다.
# @tool 데코레이터 불필요 — 시그니처(타입 힌트) + docstring 이 자동으로 도구 스키마가 된다.

from typing import Literal
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

def internet_search(
    query: str,                                                # ← 모델이 채울 필수 인자
    max_results: int = 5,                                      # ← 기본값이 있으면 모델이 보통 이 값을 그대로 씀
    topic: Literal["general", "news", "finance"] = "general",  # ← Literal → enum 제약으로 자동 변환
    include_raw_content: bool = False,
):
    """웹 검색을 실행한다"""  # ← 이 docstring 이 곧 도구의 description (모델이 보는 설명)
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

# 함수 자체가 그대로 도구 — 별도 등록 없이 create_deep_agent 의 tools= 에 넘기면 된다.
print("도구 정의 완료:", internet_search.__name__, "—", internet_search.__doc__)


도구 정의 완료: internet_search — 웹 검색을 실행한다


In [6]:
# §2.3 — create_deep_agent 호출.
#
# 이 한 호출이 LangGraph 위에 4대 능력 미들웨어 한 켜를 깔아 준다 —
# Planning(write_todos) · Filesystem · Subagents(task) · Memory.

from deepagents import create_deep_agent

research_instructions = """당신은 전문 리서처다. 당신의 임무는 철저한 조사를 수행한 뒤 깔끔한 보고서를 작성하는 것이다.
주된 정보 수집 수단으로 인터넷 검색 도구를 사용할 수 있다.

## `internet_search`

주어진 쿼리로 인터넷 검색을 실행할 때 사용한다. 반환할 결과의 최대 개수, 토픽, raw content 포함 여부를 지정할 수 있다.
"""

agent = create_deep_agent(
    model=model,                            # §2.1 의 ChatModel 인스턴스
    tools=[internet_search],                # 사용자 도구 1개 — 빌트인 9종은 자동 추가됨
    system_prompt=research_instructions,    # BASE_AGENT_PROMPT "위에" 합성됨 (교체 아님)
)

# 반환 타입은 LangGraph 의 CompiledStateGraph — .invoke / .stream / .ainvoke 모두 가능.
# create_deep_agent 가 새 추상이 아니라 LangGraph 그대로라는 게 라이브러리 설계의 핵심.
print(f"agent 타입: {type(agent).__name__}")
print(f"노드 수: {len(agent.nodes)}")


agent 타입: CompiledStateGraph
노드 수: 5


In [ ]:
# §2.4 — agent.invoke() 호출.
#
# 입력 딕셔너리 = LangGraph AgentState 스키마. 핵심 필드는 messages.
# 응답에서도 같은 dict 가 돌아오며, files / todos 슬롯도 함께 채워질 수 있다.

result = agent.invoke({
    "messages": [{"role": "user", "content": "langgraph 가 뭐야?"}]
})

# result["messages"] — 모든 turn 의 메시지 리스트 (User / AI / ToolCall / ToolMessage).
# 마지막 메시지가 모델의 최종 답변.
print("=" * 70)
print("최종 응답:")
print("=" * 70)
print(result["messages"][-1].content)

최종 응답:
LangGraph는 **LLM 애플리케이션을 그래프 구조로 설계·운영하기 위한 프레임워크**입니다.  
특히 **복잡한 에이전트 워크플로우**를 만들 때 유용합니다.

핵심은 이렇습니다:

- **노드(node)**: 작업 단계
- **엣지(edge)**: 단계 간 이동
- **상태(state)**: 실행 중 공유되는 데이터
- **조건 분기**: 상황에 따라 다음 단계가 달라짐
- **반복/루프**: 필요하면 여러 번 돌릴 수 있음

### 왜 쓰나?
일반적인 LLM 앱은 한 번 묻고 한 번 답하는 흐름이 많지만,  
LangGraph는 다음 같은 **복잡한 흐름**을 잘 다룹니다:

- 검색 → 요약 → 검증 → 재검색
- 에이전트가 도구를 쓰고 다시 판단
- 사람 검토(human-in-the-loop)
- 실패 시 재시도
- 여러 에이전트 협업

### 한 줄로 말하면
**LangChain이 LLM 앱 구성 요소를 제공한다면, LangGraph는 그 구성 요소들을 상태ful한 흐름으로 연결하는 도구**입니다.

원하면 내가 이어서  
1) **LangChain과 차이점**,  
2) **간단한 예제 코드**,  
3) **언제 LangGraph를 써야 하는지**  
중 하나로 바로 설명해드릴게요.


In [8]:
# §2.5 — 결과 인스펙션 (4대 능력의 흔적 찾기).
#
# result 안에는 최종 메시지 외에:
#   - messages : 모든 turn 의 메시지
#   - files    : write_file 로 만든 가상 파일들 (FilesystemMiddleware)
#   - todos    : write_todos 가 박은 계획 트레이스 (TodoListMiddleware)

print("=" * 70)
print(f"메시지 수: {len(result['messages'])}")
print("=" * 70)
for i, msg in enumerate(result["messages"]):
    msg_type = type(msg).__name__
    # 도구 호출이 들어 있는 AIMessage 는 짧게 요약, 그 외는 처음 100자만 미리보기.
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        names = [tc["name"] for tc in msg.tool_calls]
        preview = f"<tool_calls: {names}>"
    else:
        content = getattr(msg, "content", "")
        if isinstance(content, list):
            content = str(content)
        preview = (content[:100] + "...") if len(content) > 100 else content
    print(f"  [{i:2d}] {msg_type:20s} {preview}")

print()
print("=" * 70)
print("가상 파일시스템:")
print("=" * 70)
files = result.get("files", {})
if files:
    for path, content in files.items():
        size = len(content) if isinstance(content, str) else "?"
        print(f"  {path}  ({size}자)")
else:
    # 단순 질의는 오프로드를 발동시키지 않음. 작업형 질의("...리포트로 정리해줘")여야 발동.
    print("  (비어 있음 — 단순 질의는 오프로드를 발동시키지 않을 수 있음)")

print()
print("=" * 70)
print("Todo 리스트:")
print("=" * 70)
todos = result.get("todos", [])
if todos:
    for t in todos:
        print(f"  {t}")
else:
    print("  (비어 있음 — 짧은 질의는 write_todos 가 발동되지 않을 수 있음)")


메시지 수: 2
  [ 0] HumanMessage         langgraph 가 뭐야?
  [ 1] AIMessage            LangGraph는 **LLM 애플리케이션을 그래프 구조로 설계·운영하기 위한 프레임워크**입니다.  
특히 **복잡한 에이전트 워크플로우**를 만들 때 유용합니다.

핵심은 이렇...

가상 파일시스템:
  (비어 있음 — 단순 질의는 오프로드를 발동시키지 않을 수 있음)

Todo 리스트:
  (비어 있음 — 짧은 질의는 write_todos 가 발동되지 않을 수 있음)


## §3. Demo 2 — 모델 swap, 길 1 (문자열)

원본: [`scripts/02_model_string_swap.py`](../archives/scripts_py/02_model_string_swap.py) · 교안: §4.3 길 1

`provider:model` 문자열 한 줄로 모델을 갈아 끼우는 가장 가벼운 형태. 코드는 한 줄도 안 바꾸고 `.env` 의 `DEEPAGENT_MODEL` / `OPENAI_BASE_URL` 만 바꿔도 작동.

> **포인트**: 이 셀에는 "OpenAI"·"clipproxyapi"·"gpt-5.4-mini" 어느 문자열도 등장하지 않는다. 키 관리는 `.env` 가, 라우팅은 `OPENAI_BASE_URL` 이 맡는다.


In [9]:
# Demo 2 — Path 1 (문자열로 모델 갈아 끼우기).
#
# §2.1 의 model 이 이미 "openai:gpt-5.4-mini" 로 만들어져 있다. 여기서는
# 인자 1개 (model 만) 로 가장 작은 deep agent 를 만든다 — tools 도, system_prompt 도 안 넘기면
# BASE_AGENT_PROMPT 만으로 동작한다.

agent_demo2 = create_deep_agent(model=model)  # 인자 1개 — 정말로 "5줄로 시작" 의 가장 작은 형태

print(f"agent 타입: {type(agent_demo2).__name__}")
print(f"노드 수: {len(agent_demo2.nodes)}\n")

result_demo2 = agent_demo2.invoke({
    "messages": [{"role": "user", "content": "한 줄로 자기소개 해줘."}]
})

print("=" * 70)
print("응답:")
print("=" * 70)
print(result_demo2["messages"][-1].content)


agent 타입: CompiledStateGraph
노드 수: 5

응답:
문서, 코드, 아이디어를 빠르게 정리하고 실행까지 돕는 AI 어시스턴트입니다.


## §4. Demo 3 — 모델 swap, 길 2 (LangChain 객체 + Ollama)

원본: [`scripts/03_model_object_ollama.py`](../archives/scripts_py/03_model_object_ollama.py) · 교안: §4.3 길 2

`temperature` 같은 세부 다이얼이 필요하거나 OpenAI 호환이 아닌 백엔드(Ollama / Bedrock / Vertex)로 갈 때 객체 형식.

> ⚠️ **Ollama 모델은 tools 호출 지원이 필수** . `llama3:8b` 같은 베이스 모델은 미지원이라 deep agent 가 시작조차 못 한다. `gemma4:e4b`, `PetrosStav/gemma3-tools`, `qwen2.5:7b-instruct` 등 **tools-enabled** 모델을 써야 한다 (이 노트북은 `gemma4:e4b` 로 실행).


In [11]:
# Demo 3 — Path 2 (LangChain 모델 객체로 정밀하게).
#
# init_chat_model + provider 명시 → ChatOllama 객체가 반환된다.
# (langchain-ollama 패키지가 .venv 에 설치돼 있어야 한다)

OLLAMA_MODEL = "gemma4:e4b"  # 로컬에 받아 둔 tools-enabled 모델

model_ollama = init_chat_model(
    model=OLLAMA_MODEL,
    model_provider="ollama",  # ← 길 2 의 핵심: provider 를 "openai:" prefix 가 아닌 인자로 명시
    temperature=0,            # 결정적 응답 (같은 입력 → 같은 출력)
    # 추가 다이얼이 필요하면 여기에 num_ctx=8192, top_p=0.9 등.
)

print(f"Ollama 모델: {OLLAMA_MODEL}")
print(f"인스턴스: {type(model_ollama).__name__}")

agent_demo3 = create_deep_agent(model=model_ollama)

result_demo3 = agent_demo3.invoke({
    "messages": [{"role": "user", "content": "Ollama, 안녕?"}]
})

print()
print("=" * 70)
print("응답:")
print("=" * 70)
print(result_demo3["messages"][-1].content)


Ollama 모델: gemma4:e4b
인스턴스: ChatOllama

응답:
안녕하세요! 저는 Google에서 훈련한 대규모 언어 모델입니다.

궁금한 점이나 도움이 필요한 것이 있으신가요? 무엇이든 물어봐 주세요! 😊


## §5. Demo 4 — 커스텀 `system_prompt`

원본: [`scripts/04_custom_system_prompt.py`](../archives/scripts_py/04_custom_system_prompt.py) · 교안: §4.4

`system_prompt=` 한 인자로 도메인 한 장을 얹는다. BASE 를 **교체** 하는 게 아니라 **앞에 prepend** 된다 — `USER → BASE → SUFFIX` 3단 합성의 USER 자리.


In [11]:
# Demo 4 — 가장 작은 커스텀 system_prompt.
#
# Quickstart(Demo 1) 의 길고 친절한 prompt 가 도메인을 좁히는 짧은 한 장으로 줄어든 형태일 뿐이다.

# 백슬래시(\) 줄 잇기 — 트리플 쿼트 안에서도 동작한다.
# 줄 끝 \ 가 다음 줄로 이어 붙어 결과 문자열에 줄바꿈이 들어가지 않는다.
research_instructions_short = """\
당신은 전문 리서처다. 당신의 임무는 철저한 \
조사를 수행한 뒤 깔끔한 보고서를 작성하는 것이다. \
"""

agent_demo4 = create_deep_agent(
    model=model,                                # §2.1 의 model 재사용
    system_prompt=research_instructions_short,  # USER 슬롯에 들어가서 BASE 위에 합성됨
)

result_demo4 = agent_demo4.invoke({
    "messages": [
        {"role": "user", "content": "RAG 와 fine-tuning 의 차이를 한 단락으로."}
    ]
})

print("=" * 70)
print("응답 (research_instructions_short 기조로 답변):")
print("=" * 70)
print(result_demo4["messages"][-1].content)


응답 (research_instructions_short 기조로 답변):
RAG(Retrieval-Augmented Generation)는 모델이 답을 생성하기 전에 외부 문서나 데이터베이스에서 관련 정보를 검색해 그 내용을 바탕으로 응답하는 방식이고, fine-tuning은 기존 모델의 파라미터를 추가 학습시켜 특정 도메인이나 작업에 맞게 모델 자체를 바꾸는 방식이다. 쉽게 말해 RAG는 “필요할 때 참고자료를 찾아 쓰는 것”에 가깝고, fine-tuning은 “모델의 습관과 지식을 직접 바꾸는 것”에 가깝다. 그래서 RAG는 최신 정보 반영과 근거 제시에 유리하고, fine-tuning은 특정 스타일·형식·업무 규칙을 안정적으로 익히게 하는 데 강하다.


In [12]:
# 보너스 — BASE_AGENT_PROMPT 실체 들여다보기.
#
# 위 research_instructions_short 가 USER 자리에 들어가고, 그 뒤에 합성되는 BASE 의 정체.
# 이 ~42줄 텍스트가 Understand → Act → Verify 워크플로와 도구 사용 정책을 모델에 박아 준다.

from deepagents.graph import BASE_AGENT_PROMPT

print("BASE_AGENT_PROMPT 길이:", len(BASE_AGENT_PROMPT), "자")
print("=" * 70)
print(BASE_AGENT_PROMPT)


BASE_AGENT_PROMPT 길이: 2257 자
You are a deep agent, an AI assistant that helps users accomplish tasks using tools. You respond with text and tool calls. The user can see your responses and tool outputs in real time.

## Core Behavior

- Be concise and direct. Don't over-explain unless asked.
- NEVER add unnecessary preamble ("Sure!", "Great question!", "I'll now...").
- Don't say "I'll now do X" — just do it.
- If the request is underspecified, ask only the minimum followup needed to take the next useful action.
- If asked how to approach something, explain first, then act.

## Professional Objectivity

- Prioritize accuracy over validating the user's beliefs
- Disagree respectfully when the user is incorrect
- Avoid unnecessary superlatives, praise, or emotional validation

## Doing Tasks

When the user asks you to do something:

1. **Understand first** — read relevant files, check existing patterns. Quick but thorough — gather enough evidence to start, then iterate.
2. **Act** — imple

**한국어 대조판 (학습용)** — 위 셀의 영문 BASE_AGENT_PROMPT 를 줄 단위로 옮긴 것. 실제 라이브러리는 영문 그대로 모델에 주입되며, 아래 번역은 의미 파악·디버깅용 참고다.

```markdown
너는 deep agent — 도구를 써서 사용자의 작업을 돕는 AI 어시스턴트다. 텍스트와 도구 호출로 응답하며, 사용자는 너의 응답과 도구 출력을 실시간으로 본다.

## 핵심 행동 양식 (Core Behavior)

- 간결하고 직접적으로 답하라. 묻지 않은 것까지 길게 설명하지 마라.
- 불필요한 서두("좋습니다!", "좋은 질문이네요!", "이제 ...하겠습니다") 를 절대 붙이지 마라.
- "이제 X 를 하겠다" 고 말하지 말고 — 그냥 해라.
- 요청이 모호하면, 다음 유용한 행동을 취하는 데 필요한 **최소한의** 후속 질문만 던져라.
- 접근 방식을 묻는 질문이면 설명 먼저, 그다음 실행하라.

## 전문가적 객관성 (Professional Objectivity)

- 사용자의 믿음을 받쳐주는 것보다 **정확성** 을 우선하라.
- 사용자가 틀렸을 때는 정중하게 반박하라.
- 불필요한 최상급 표현, 칭찬, 감정적 동의는 피하라.

## 작업 수행 (Doing Tasks)

사용자가 어떤 일을 요청하면:

1. **먼저 이해하라** — 관련 파일을 읽고, 기존 패턴을 확인하라. 빠르되 충실하게 — 시작에 필요한 만큼의 근거를 모은 뒤 반복적으로 진행하라.
2. **실행하라** — 해결책을 구현하라. 빠르게, 그러나 정확하게.
3. **검증하라** — 너의 출력이 아니라 요청받은 내용을 기준으로 결과를 확인하라. 첫 시도가 옳은 경우는 드물다 — 반복하라.

작업이 완전히 끝날 때까지 계속 진행하라. 도중에 멈춰서 "이렇게 하겠다" 고 설명하지 말고 — 그냥 해라. 작업이 완료됐거나 진짜로 막혔을 때에만 사용자에게 제어를 넘겨라.

**문제가 생기면:**

- 같은 시도가 반복적으로 실패하면 멈추고 *왜* 그런지 분석하라 — 같은 방식을 계속 재시도하지 마라.
- 막혔다면, 무엇이 잘못됐는지 사용자에게 알리고 가이드를 요청하라.

## 명확화 질문 (Clarifying Requests)

- 사용자가 이미 제공한 세부 사항은 묻지 마라.
- 요청에 명백히 함의된 내용이라면 합리적인 기본값을 사용하라.
- 누락된 의미 항목 — 내용, 전달 방식, 상세도, 알림 기준 — 을 우선 물어라.
- 짧은 막힘 해소형 질문 하나로 작업이 진척될 수 있다면, 도구·스케줄링·통합의 한계를 길게 설명하며 시작하지 마라.
- 구현 관련 질문보다 **도메인 정의** 질문을 먼저 던져라.
- 모니터링·알림 요청에는 어떤 신호·임계값·조건에서 알림을 발동할지 물어라.

## 진행 보고 (Progress Updates)

긴 작업에서는 적절한 간격으로 짧은 진행 보고를 남겨라 — 지금까지 한 일과 다음에 할 일을 한 문장으로 요약하라.
```


## §6. (옵션) LangSmith 트레이싱

env 변수 3개만 채우면 다음 `agent.invoke(...)` 부터 자동으로 LangSmith UI 에 trace 가 쌓인다.


In [13]:
# (실행 안 함) LangSmith 트레이싱 활성화 시범.
#
# 아래 두 줄을 .env 에 추가하기만 하면 다음 invoke 부터 자동 추적된다.
# (실제 키가 없으면 LangSmith 가 조용히 무시하므로 setenv 만 보여준다.)

import os

os.environ.setdefault("LANGCHAIN_TRACING_V2", "false")  # true 로 두면 추적 시작
os.environ.setdefault("LANGCHAIN_API_KEY", "")          # ls__... 로 시작하는 LangSmith 키
os.environ.setdefault("LANGCHAIN_PROJECT", "week1-walkthrough")  # 프로젝트 라벨

print(f"LANGCHAIN_TRACING_V2 = {os.environ['LANGCHAIN_TRACING_V2']}")
print(f"LANGCHAIN_PROJECT    = {os.environ['LANGCHAIN_PROJECT']}")
print(f"API_KEY 설정?         = {bool(os.environ['LANGCHAIN_API_KEY'])}")
print()
print("→ 위 세 값이 모두 설정된 상태에서 agent.invoke(...) 를 호출하면,")
print("   LangSmith UI 의 'week1-walkthrough' 프로젝트에 trace 가 쌓인다.")


LANGCHAIN_TRACING_V2 = false
LANGCHAIN_PROJECT    = week1-walkthrough
API_KEY 설정?         = False

→ 위 세 값이 모두 설정된 상태에서 agent.invoke(...) 를 호출하면,
   LangSmith UI 의 'week1-walkthrough' 프로젝트에 trace 가 쌓인다.
